# Bilig WorkPaper Formula Readback With CrewAI

This quickstart shows a CrewAI tool pattern for spreadsheet-backed business logic. The tool writes one input cell in a Bilig WorkPaper forecast model, recalculates dependent formulas, verifies readback, and returns a compact proof object.

Use this when an agent needs spreadsheet logic without driving Excel, Google Sheets, or a browser.

## Install Dependencies

The deterministic smoke test only calls the Bilig API. The CrewAI agent run also needs an LLM key such as `OPENAI_API_KEY`.

In [ ]:
%pip install -U -q crewai pydantic

## Define The Bilig WorkPaper Tool

The tool fails closed: if Bilig does not return `verified: true`, the call raises instead of letting the agent continue with stale or guessed formula values.

In [ ]:
from __future__ import annotations

import json
import os
from typing import Any
from urllib.parse import urljoin
from urllib.request import Request, urlopen

from crewai import Agent, Crew, Process, Task
from crewai.tools import BaseTool
from pydantic import BaseModel, Field


DEFAULT_BASE_URL = "https://bilig.proompteng.ai"
DEFAULT_SHEET_NAME = "Inputs"
DEFAULT_ADDRESS = "B3"
DEFAULT_VALUE = 0.4


class ForecastFormulaInput(BaseModel):
    sheet_name: str = Field(default=DEFAULT_SHEET_NAME, description="Worksheet containing the editable forecast input.")
    address: str = Field(default=DEFAULT_ADDRESS, description="A1-style input cell address to edit before recalculation.")
    value: float = Field(default=DEFAULT_VALUE, description="Numeric value to write before formula readback.")


class BiligWorkPaperForecastTool(BaseTool):
    name: str = "Bilig WorkPaper forecast formula readback"
    description: str = (
        "Edit one forecast workbook input, recalculate formulas, and return a "
        "verified before/after readback proof from Bilig WorkPaper."
    )
    args_schema: type[BaseModel] = ForecastFormulaInput
    base_url: str = Field(default=DEFAULT_BASE_URL)
    timeout: int = Field(default=30)

    def _run(self, sheet_name: str = DEFAULT_SHEET_NAME, address: str = DEFAULT_ADDRESS, value: float = DEFAULT_VALUE) -> str:
        proof = call_bilig_forecast(
            base_url=self.base_url,
            sheet_name=sheet_name,
            address=address,
            value=value,
            timeout=self.timeout,
        )
        return json.dumps(compact_proof(proof), indent=2)


def call_bilig_forecast(*, base_url: str, sheet_name: str, address: str, value: float, timeout: int) -> dict[str, Any]:
    if not base_url.startswith(("http://", "https://")):
        raise ValueError("BILIG_BASE_URL must start with http:// or https://")

    endpoint = urljoin(base_url.rstrip("/") + "/", "api/workpaper/n8n/forecast")
    body = json.dumps({"sheetName": sheet_name, "address": address.upper(), "value": value}).encode("utf-8")
    request = Request(
        endpoint,
        data=body,
        headers={
            "accept": "application/json",
            "content-type": "application/json",
            "user-agent": "CrewAI-Bilig-WorkPaper-Quickstart/0.1",
        },
        method="POST",
    )

    with urlopen(request, timeout=timeout) as response:
        parsed = json.loads(response.read().decode("utf-8"))

    if not isinstance(parsed, dict):
        raise ValueError(f"Expected JSON object response, received {type(parsed).__name__}")
    if parsed.get("verified") is not True:
        raise ValueError(f"Bilig returned an unverified WorkPaper response: {parsed}")

    return parsed


def compact_proof(proof: dict[str, Any]) -> dict[str, Any]:
    before = proof.get("before") if isinstance(proof.get("before"), dict) else {}
    after = proof.get("after") if isinstance(proof.get("after"), dict) else {}
    checks = proof.get("checks") if isinstance(proof.get("checks"), dict) else {}

    return {
        "verified": proof.get("verified") is True,
        "editedCell": proof.get("editedCell"),
        "before": {"expectedArr": before.get("expectedArr"), "targetGap": before.get("targetGap")},
        "after": {"expectedArr": after.get("expectedArr"), "targetGap": after.get("targetGap")},
        "checks": {
            "formulasPersisted": checks.get("formulasPersisted") is True,
            "restoredMatchesAfter": checks.get("restoredMatchesAfter") is True,
            "computedOutputChanged": checks.get("computedOutputChanged") is True,
        },
        "source": "Bilig WorkPaper",
        "github": "https://github.com/proompteng/bilig",
    }

## Run A Deterministic Tool Smoke Test

This cell validates the workbook API and proof object without using an LLM.

In [ ]:
base_url = os.getenv("BILIG_BASE_URL", DEFAULT_BASE_URL)
tool = BiligWorkPaperForecastTool(base_url=base_url)

print(tool._run(sheet_name="Inputs", address="B3", value=0.4))

## Let A CrewAI Agent Use The Tool

Run this section after setting your model provider key, for example `OPENAI_API_KEY`. The agent is instructed to answer only from computed readback returned by the tool.

In [ ]:
verifier = Agent(
    role="Workbook formula verifier",
    goal="Use verified workbook readback to explain whether the edited forecast input changed the computed ARR output.",
    backstory=(
        "You check spreadsheet-backed business logic through API tools. "
        "You do not infer formula results; you rely on computed readback."
    ),
    tools=[tool],
    verbose=True,
)

task = Task(
    description=(
        "Set the forecast input cell to 0.4 with the Bilig WorkPaper tool. "
        "Report the edited cell, before and after expected ARR, target gap, "
        "and whether formulas persisted after restore."
    ),
    expected_output="A concise verification summary grounded only in the Bilig WorkPaper proof object returned by the tool.",
    agent=verifier,
)

crew = Crew(agents=[verifier], tasks=[task], process=Process.sequential, verbose=True)
result = crew.kickoff()
print(result)